# Notebook 04 — One Operator for All Days

NB03 fits one network per day (~90 s/day). This notebook trains **neural operators** that map a raw quote cloud straight to a smoothed surface — trained once, applied to every day:

- **DeepONet** (set encoder + trunk) and a **graph neural operator (GNO)**, each under three regimes: **R1** real-only, **R2** synthetic-only, **R3** synthetic pretrain + real fine-tune;
- the **prior-embedded** v3 variants (section below), which wrap each operator around a per-day GJ-certified SSVI prior.

Every model is scored with the NB03 methodology — penalty-grid view vs **independent audit grid**, strict vs **material** violation rates, blind-spot flags, fit-gated aggregates — plus two operator-specific measurements: **discretization invariance** and **per-surface latency**. Dense surface packs are exported for NB05.

## 0. Imports & configuration

Everything is env-driven (`NB04_*`), so the cluster scripts can rescale the run without touching the notebook.

In [ ]:
import os
import json
import time
import copy
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

torch.set_default_dtype(torch.float32)
np.set_printoptions(precision=4, suppress=True)
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook_connected")

In [ ]:
def default_clean_dir():
    env = os.environ.get("THESIS_OUT_DIR")
    if env:
        return Path(env)
    for candidate in [Path("data/clean"), Path("main/data/clean")]:
        if (candidate / "option_prices_clean.parquet").exists():
            return candidate
    return Path("data/clean")


OUT_DIR = default_clean_dir()
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_DIR / "nb04_operator_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
REAL_PARQUET = Path(os.environ.get("THESIS_OPT_PARQUET", str(OUT_DIR / "option_prices_clean.parquet")))

In [ ]:
SEED = int(os.environ.get("NB04_SEED", "0"))
DEVICE = torch.device(os.environ.get("NB04_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))

# Defaults are intentionally serious for CPU. Increase for final overnight runs.
N_SYNTH = int(os.environ.get("NB04_N_SYNTH", "1000"))
LIMIT_DAYS_RAW = os.environ.get("NB04_LIMIT_DAYS", "None")
LIMIT_DAYS = None if LIMIT_DAYS_RAW in ("", "None", "none", "ALL", "all") else int(LIMIT_DAYS_RAW)

EPOCHS_PRE = int(os.environ.get("NB04_EPOCHS_PRE", "25"))
EPOCHS_R1 = int(os.environ.get("NB04_EPOCHS_R1", "25"))
EPOCHS_FT = int(os.environ.get("NB04_EPOCHS_FT", "12"))

MAX_QUOTES_LOAD = int(os.environ.get("NB04_MAX_QUOTES_PER_DAY", "2500"))
FIT_POINTS_DEEP = int(os.environ.get("NB04_FIT_POINTS_DEEP", "512"))
INPUT_POINTS_DEEP = int(os.environ.get("NB04_INPUT_POINTS_DEEP", "900"))
FIT_POINTS_GNO = int(os.environ.get("NB04_FIT_POINTS_GNO", "160"))
INPUT_POINTS_GNO = int(os.environ.get("NB04_INPUT_POINTS_GNO", "220"))

BATCH_DEEP = int(os.environ.get("NB04_BATCH_DEEP", "8"))
BATCH_GNO = int(os.environ.get("NB04_BATCH_GNO", "1"))

LATENT = int(os.environ.get("NB04_LATENT", "64"))
ENC_H = int(os.environ.get("NB04_ENC_H", "128"))
TRUNK_H = int(os.environ.get("NB04_TRUNK_H", "128"))
GNO_CHANNELS = int(os.environ.get("NB04_GNO_CHANNELS", "8"))
GNO_LAYERS = int(os.environ.get("NB04_GNO_LAYERS", "2"))
GNO_K = int(os.environ.get("NB04_GNO_K", "16"))
GNO_RHO = float(os.environ.get("NB04_GNO_RHO", "0.35"))

GRID_NK = int(os.environ.get("NB04_GRID_NK", "22"))
GRID_NT = int(os.environ.get("NB04_GRID_NT", "9"))
EPS_BUT = 1e-3
LAMBDAS = dict(fit=1.0, but=10.0, cal=10.0, reg_t=0.01, reg_k=0.01)

### The independent audit grid

In v1 the penalties and the reported violation metrics lived on the *same* 22×9 grid: the audit could not, by construction, see anything the penalty missed — the NB03-v3 pathology, verbatim. The audit grid is therefore finer, uniform and disjoint from the penalty grid, and violations are reported **strict** (< −1e−8) and **material** (< −`VIOL_TOL_MAT`, monetizable against quote noise).

In [ ]:
AUDIT_NK = int(os.environ.get("NB04_AUDIT_NK", "61"))
AUDIT_NT = int(os.environ.get("NB04_AUDIT_NT", "31"))
VIOL_TOL_MAT = 1e-3
HOLDOUT_FRAC = 0.20            # per-day holdout, NB03 protocol: comparable numbers across notebooks
RESUME = os.environ.get("NB04_RESUME", "0") == "1"   # reload saved state dicts, skip training
FIT_GATE_VP = 5.0              # invariance/audit metrics are vacuous on a model that cannot fit

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

cfg = {k: v for k, v in list(globals().items()) if k.startswith("NB04_")}
print("REAL_PARQUET:", REAL_PARQUET.resolve())
print("OUT_DIR     :", OUT_DIR.resolve())
print("RUN_DIR     :", RUN_DIR.resolve())
print("DEVICE      :", DEVICE)
print("all days    :", LIMIT_DAYS is None)
print("epochs      :", dict(pretrain=EPOCHS_PRE, real=EPOCHS_R1, finetune=EPOCHS_FT))

## 1. Data — real SPX bank and synthetic SSVI bank

The domain is a fixed box in $(k,\tau)$. Synthetic surfaces are sampled from SSVI with random parameters; real days come from the NB01 clean parquet, subsampled to a bounded per-day quote budget with a `crc32(date)` seed so the draw is a pure function of the date.

In [ ]:
K_LO, K_HI = -0.50, 0.35
T_LO, T_HI = 0.05, 1.50


def ssvi_total_variance(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def sample_ssvi_params(r):
    return dict(
        rho=float(r.uniform(-0.80, -0.10)),
        eta=float(r.uniform(0.40, 1.45)),
        gamma=float(r.uniform(0.18, 0.50)),
        alpha=float(r.uniform(0.015, 0.12)),
        beta=float(r.uniform(0.75, 1.20)),
    )


def synth_surface(r, params=None, n_slices=(5, 10), n_per=(8, 24), noise=0.012):
    p = params or sample_ssvi_params(r)
    taus = np.sort(r.uniform(T_LO, T_HI, r.integers(n_slices[0], n_slices[1] + 1)))
    ks, ts, ivs = [], [], []
    for tau in taus:
        n = int(r.integers(n_per[0], n_per[1] + 1))
        k = np.sort(r.uniform(K_LO, K_HI, n))
        theta = p["alpha"] * tau ** p["beta"]
        w = ssvi_total_variance(k, theta, p["rho"], p["eta"], p["gamma"])
        iv = np.sqrt(np.maximum(w / tau, 1e-10)) * (1 + noise * r.standard_normal(n))
        ks.append(k); ts.append(np.full(n, tau)); ivs.append(iv)
    return dict(date="synthetic", k=np.concatenate(ks), tau=np.concatenate(ts), iv=np.concatenate(ivs), kind="synthetic")

In [ ]:
def bounded_sample(k, tau, iv, max_n, r):
    ok = (
        np.isfinite(k) & np.isfinite(tau) & np.isfinite(iv)
        & (k >= K_LO) & (k <= K_HI)
        & (tau >= T_LO) & (tau <= T_HI)
        & (iv > 0) & (iv < 5)
    )
    k, tau, iv = k[ok], tau[ok], iv[ok]
    if len(k) > max_n:
        idx = r.choice(len(k), size=max_n, replace=False)
        k, tau, iv = k[idx], tau[idx], iv[idx]
    order = np.lexsort((k, tau))
    return k[order], tau[order], iv[order]

In [ ]:
def load_real_bank():
    assert REAL_PARQUET.exists(), f"Missing clean parquet: {REAL_PARQUET}. Run NB01 first."
    cols = pl.scan_parquet(REAL_PARQUET).collect_schema().names()
    iv_col = "iv_om" if "iv_om" in cols else "impl_volatility"
    filters = (
        pl.col("is_otm")
        & pl.col(iv_col).is_not_null()
        & pl.col("k").is_not_null()
        & pl.col("tau").is_not_null()
    )
    if "discount_valid" in cols:
        filters = filters & pl.col("discount_valid")
    scan = pl.scan_parquet(REAL_PARQUET).filter(filters)
    dates = scan.select("date").unique().collect(engine="streaming")["date"].sort().to_list()
    if LIMIT_DAYS is not None:
        dates = dates[-LIMIT_DAYS:]
    df = (
        scan.filter(pl.col("date").is_in(dates))
        .select(["date", "tau", "k", iv_col])
        .rename({iv_col: "iv"})
        .collect(engine="streaming")
        .sort("date")
    )

    import zlib
    days = []
    for d in dates:
        sub = df.filter(pl.col("date") == d)
        # v3: per-day seed. With a single sequential rng the subsample of a given day
        # depended on LIMIT_DAYS (how many days precede it in the loop): staging-run and
        # production-run metrics were not comparable day by day. crc32(date) makes the
        # day's 2500-quote draw a pure function of the date, as the holdout already is.
        r = np.random.default_rng(zlib.crc32(str(d).encode()) + SEED)
        k, tau, iv = bounded_sample(
            sub["k"].to_numpy(),
            sub["tau"].to_numpy(),
            sub["iv"].to_numpy(),
            MAX_QUOTES_LOAD,
            r,
        )
        if len(k) >= 40:
            days.append(dict(date=str(d), k=k, tau=tau, iv=iv, kind="real"))
    return days

In [ ]:
real_bank = load_real_bank()
synth_bank = [synth_surface(rng) for _ in range(N_SYNTH)]

n = len(real_bank)
i_tr, i_va = int(0.60 * n), int(0.80 * n)
train_days, val_days, test_days = real_bank[:i_tr], real_bank[i_tr:i_va], real_bank[i_va:]

def size_summary(bank):
    sizes = np.array([len(s["k"]) for s in bank])
    return dict(n=len(bank), min=int(sizes.min()), median=int(np.median(sizes)), max=int(sizes.max()))

print("real bank      :", size_summary(real_bank), real_bank[0]["date"], "->", real_bank[-1]["date"])
print("train/val/test :", len(train_days), len(val_days), len(test_days))
print("synthetic bank :", size_summary(synth_bank))
assert train_days and val_days and test_days, "Chronological split produced an empty subset."

## 2. Grids, weights and audit machinery

In [ ]:
import zlib

def as_t(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)


def sample_idx(n, m, r):
    if m is None or n <= m:
        return np.arange(n)
    return np.sort(r.choice(n, size=m, replace=False))


def scale_coords_t(k, tau):
    k = (k - (K_LO + K_HI) / 2) / ((K_HI - K_LO) / 2)
    tau = (tau - (T_LO + T_HI) / 2) / ((T_HI - T_LO) / 2)
    return torch.stack([k, tau], dim=-1)


KG = np.linspace(K_LO + 0.02, K_HI - 0.02, GRID_NK)
TG = np.linspace(T_LO + 0.02, T_HI - 0.02, GRID_NT)
KKG, TTG = np.meshgrid(KG, TG)
KG_FLAT, TG_FLAT = KKG.ravel(), TTG.ravel()
DK = float(KG[1] - KG[0])
DT = float(TG[1] - TG[0])
TTG_T = as_t(TTG)
KKG_T = as_t(KKG)

In [ ]:
def durrleman_g_np(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def durrleman_g_t(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def vega_weights(k, tau, iv):
    d1 = (-k + 0.5 * iv ** 2 * tau) / (iv * torch.sqrt(tau))
    vega = torch.exp(-0.5 * d1 ** 2) / np.sqrt(2 * np.pi) * torch.sqrt(tau)
    w = vega / torch.clamp(vega.mean(), min=1e-8)
    return torch.clamp(w, max=25.0)

In [ ]:
# penalty grid (used inside the loss -- unchanged, deliberately: the GAP between what
# this grid sees and what the independent audit sees is itself a thesis measurement)
def fd_audit_from_grid(V):
    """Penalty-grid view: what the optimizer saw. v3: calendar in DERIVATIVE units
    (d_tau w = delta W / delta tau) with strict AND material thresholds. v2 compared a
    raw-difference strict zero (penalty side) to a derivative material rate (audit side):
    the blind-spot comparison was asymmetric and under-counted calendar blind days."""
    W = V ** 2 * TTG
    Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
    Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
    G = durrleman_g_np(KKG[:, 1:-1], W[:, 1:-1], Wk, Wkk)
    WT = (W[1:, :] - W[:-1, :]) / DT
    return dict(
        min_g=float(G.min()),
        min_dtw=float(WT.min()),
        butterfly_viol_pct=float(np.mean(G < -1e-8) * 100),
        calendar_viol_pct=float(np.mean(WT < -1e-8) * 100),
        bfly_viol_mat_pct=float(np.mean(G < -VIOL_TOL_MAT) * 100),
        cal_viol_mat_pct=float(np.mean(WT < -VIOL_TOL_MAT) * 100),
    )

In [ ]:
# --- INDEPENDENT audit grid (finer, uniform, disjoint nodes) ---
KA = np.linspace(K_LO + 0.015, K_HI - 0.015, AUDIT_NK)
TA = np.linspace(T_LO + 0.015, T_HI - 0.015, AUDIT_NT)
KKA, TTA = np.meshgrid(KA, TA)
KA_FLAT, TA_FLAT = KKA.ravel(), TTA.ravel()
DKA = float(KA[1] - KA[0])
DTA = float(TA[1] - TA[0])


def hull_mask_for(surf):
    """v3 [N4-1]: the day's QUOTED domain on the audit grid. For each audit tau, the quoted
    k-range interpolated between the day's actual maturities; audit taus outside the day's
    quoted maturity span are out of hull. The box audit answers 'is the surface arbitrage-free
    where a production system would query it'; the hull audit answers 'is it arbitrage-free
    where the data actually was'. The thesis reports BOTH (full box / quoted domain), same
    convention as every other notebook."""
    taus = np.unique(surf["tau"])
    kmin = np.array([float(surf["k"][surf["tau"] == t].min()) for t in taus])
    kmax = np.array([float(surf["k"][surf["tau"] == t].max()) for t in taus])
    lo = np.interp(TA, taus, kmin)
    hi = np.interp(TA, taus, kmax)
    in_t = (TA >= taus.min() - 1e-12) & (TA <= taus.max() + 1e-12)
    return (KKA >= lo[:, None]) & (KKA <= hi[:, None]) & in_t[:, None]


def _masked_stats(A, M, tol):
    """(min, strict %, material %) of A restricted to mask M; NaN triple if M is empty."""
    if not M.any():
        return float("nan"), float("nan"), float("nan")
    a = A[M]
    return (float(a.min()),
            float(np.mean(a < -1e-8) * 100),
            float(np.mean(a < -tol) * 100))

In [ ]:
def independent_audit(V, hull=None):
    """Strict + material butterfly/calendar rates on the fine audit grid, on the full box
    and (if a hull mask is supplied) on the day's quoted domain."""
    W = V ** 2 * TTA
    Wk = (W[:, 2:] - W[:, :-2]) / (2 * DKA)
    Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DKA ** 2)
    G = durrleman_g_np(KKA[:, 1:-1], W[:, 1:-1], Wk, Wkk)
    WT = (W[1:, :] - W[:-1, :]) / DTA
    out = dict(
        audit_min_g=float(G.min()),
        audit_min_dtw=float(WT.min()),
        audit_bfly_viol_pct=float(np.mean(G < -1e-8) * 100),
        audit_cal_viol_pct=float(np.mean(WT < -1e-8) * 100),
        audit_bfly_viol_mat_pct=float(np.mean(G < -VIOL_TOL_MAT) * 100),
        audit_cal_viol_mat_pct=float(np.mean(WT < -VIOL_TOL_MAT) * 100),
    )
    if hull is not None:
        mg = hull[:, 1:-1]                    # G lives on interior k nodes
        mt = hull[1:, :] & hull[:-1, :]       # WT lives between tau nodes
        g_min, _, g_mat = _masked_stats(G, mg, VIOL_TOL_MAT)
        t_min, _, t_mat = _masked_stats(WT, mt, VIOL_TOL_MAT)
        out.update(
            hull_frac=float(hull.mean()),
            hull_min_g=g_min,
            hull_min_dtw=t_min,
            hull_bfly_viol_mat_pct=g_mat,
            hull_cal_viol_mat_pct=t_mat,
        )
    return out


def blind_flags(pg, ia):
    """v3 [N4-2]: symmetric, material-vs-material on BOTH constraints. Box audit on both
    sides: the penalty grid spans the same box, so box-vs-box is the fair comparison for
    the blind-spot mechanism (severity within the quoted domain is the hull_* columns)."""
    return dict(
        blind_bfly=bool(pg["min_g"] > -VIOL_TOL_MAT
                        and ia["audit_min_g"] < -VIOL_TOL_MAT),
        blind_cal=bool(pg["cal_viol_mat_pct"] == 0.0
                       and ia["audit_cal_viol_mat_pct"] > 0.0),
    )

In [ ]:
# Validate finite differences once against an SSVI closed form.
p0 = dict(rho=-0.5, eta=0.9, gamma=0.4)
theta0 = 0.04 * 0.5
phi0 = p0["eta"] * theta0 ** (-p0["gamma"])
w = ssvi_total_variance(KG, theta0, **p0)
sq = np.sqrt((phi0 * KG + p0["rho"]) ** 2 + 1 - p0["rho"] ** 2)
wp = 0.5 * theta0 * (p0["rho"] * phi0 + phi0 * (phi0 * KG + p0["rho"]) / sq)
wpp = 0.5 * theta0 * phi0 ** 2 * (1 - p0["rho"] ** 2) / sq ** 3
wk_fd = (w[2:] - w[:-2]) / (2 * DK)
wkk_fd = (w[2:] - 2 * w[1:-1] + w[:-2]) / (DK ** 2)
err = np.max(np.abs(durrleman_g_np(KG[1:-1], w[1:-1], wk_fd, wkk_fd) - durrleman_g_np(KG, w, wp, wpp)[1:-1]))
print(f"FD Durrleman validation max error: {err:.2e}")
assert err < 3e-2, "Finite-difference grid is too coarse; increase NB04_GRID_NK."

## 3. The two operator families

**DeepONet**: a permutation-invariant set encoder pools the quote cloud into a latent code; a trunk net evaluates the surface at arbitrary $(k,\tau)$. **GNO**: kernel message passing from input quotes to query points, restricted to a $\tau$-neighborhood.

In [ ]:
class MLP(nn.Module):
    def __init__(self, sizes, activation=nn.Tanh):
        super().__init__()
        layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            layers.append(nn.Linear(a, b))
            if i < len(sizes) - 2:
                layers.append(activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
class DeepSetONet(nn.Module):
    model_name = "deeponet"
    fit_points = FIT_POINTS_DEEP
    input_points = INPUT_POINTS_DEEP
    batch_size = BATCH_DEEP

    def __init__(self):
        super().__init__()
        self.encoder = MLP([3, ENC_H, ENC_H, LATENT])
        self.psi = MLP([LATENT, ENC_H, LATENT])
        self.trunk = MLP([2, TRUNK_H, TRUNK_H, LATENT])
        # v2: initialize the output bias so softplus(bias) ~ 0.20 (a typical IV level). v1 started
        # at softplus(0)=0.69 and, on real data (heavy far-OTM tails, exploding vega weights), the
        # pre-activation was driven far negative early -- softplus gradient vanishes and the model
        # DIES at pred~0: the R1 run sat at loss = 1.0000 exactly (the value of the relative loss
        # for a zero prediction) with min_g = 0.9995 (a k-flat surface) for 20 epochs.
        self.bias = nn.Parameter(torch.tensor(float(np.log(np.expm1(0.20)))))

    def encode_surface(self, surf, input_idx):
        k = as_t(surf["k"][input_idx])
        tau = as_t(surf["tau"][input_idx])
        iv = as_t(surf["iv"][input_idx])
        x = torch.cat([scale_coords_t(k, tau), 5.0 * iv[:, None]], dim=-1)
        return self.encoder(x).mean(dim=0)

    def predict_surface(self, surf, kq, tq, input_idx):
        z = self.encode_surface(surf, input_idx)
        coords = scale_coords_t(as_t(kq), as_t(tq))
        c = self.psi(z[None, :])[0]
        t = self.trunk(coords)
        return F.softplus((t * c).sum(dim=-1) + self.bias) + 1e-6

In [ ]:
def ffn(sizes, activation=nn.GELU):
    layers = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)


def build_neighbors(x_in, y_out, rho_bar=GNO_RHO, K=GNO_K):
    nbrs = []
    for y in y_out:
        mask = (x_in[:, 1] - y[1]).abs() <= rho_bar
        idx = torch.nonzero(mask, as_tuple=False).flatten()
        if idx.numel() == 0:
            idx = torch.arange(len(x_in), device=x_in.device)
        d = ((x_in[idx] - y) ** 2).sum(-1)
        idx = idx[d.argsort()]
        stride = max(1, int(np.ceil(idx.numel() / K)))
        nbrs.append(idx[::stride][:K])
    return nbrs

In [ ]:
class GNOLayer(nn.Module):
    def __init__(self, c_in, c_out, local=True, hidden=64):
        super().__init__()
        self.P = ffn([c_in, hidden, c_in])
        self.kW = ffn([4 + c_in + 1, hidden, hidden, c_out * c_in])
        self.kb = ffn([4 + c_in + 1, hidden, hidden, c_out])
        self.W = nn.Linear(c_in, c_out) if local else None
        self.Q = ffn([c_out, hidden, c_out])
        self.act = nn.GELU()
        self.c_in, self.c_out = c_in, c_out

    def forward(self, h_in, x_in, y_out, v_in, nbrs, h_self=None):
        h_t = self.P(h_in)
        out = []
        for j, y in enumerate(y_out):
            idx = nbrs[j]
            xz, hz, vz = x_in[idx], h_t[idx], v_in[idx]
            feat = torch.cat([y.expand(len(idx), 2), xz, hz, vz], dim=-1)
            Wk = self.kW(feat).view(len(idx), self.c_out, self.c_in)
            bk = self.kb(feat)
            out.append((torch.einsum("nij,nj->ni", Wk, hz) + bk).mean(dim=0))
        out = torch.stack(out)
        if self.W is not None and h_self is not None:
            out = out + self.W(self.P(h_self))
        return self.act(self.Q(out))

In [ ]:
class InterpGNO(nn.Module):
    model_name = "gno"
    fit_points = FIT_POINTS_GNO
    input_points = INPUT_POINTS_GNO
    batch_size = BATCH_GNO

    def __init__(self, channels=GNO_CHANNELS, J=GNO_LAYERS):
        super().__init__()
        self.lift = ffn([1, 64, channels])
        self.layers = nn.ModuleList([GNOLayer(channels, channels, local=(j > 0)) for j in range(J)])
        self.out = nn.Sequential(nn.Linear(channels, 1), nn.Softplus())

    def forward_coords(self, x_in, v_in, y_out):
        nbrs = build_neighbors(x_in, y_out)
        h_in = self.lift(v_in)
        h = self.layers[0](h_in, x_in, y_out, v_in, nbrs)
        nbrs_out = build_neighbors(y_out, y_out)
        v_zero = torch.zeros(len(y_out), 1, device=y_out.device)
        for layer in self.layers[1:]:
            h = layer(h, y_out, y_out, v_zero, nbrs_out, h_self=h)
        return self.out(h).squeeze(-1) + 1e-6

    def predict_surface(self, surf, kq, tq, input_idx):
        kin = as_t(surf["k"][input_idx])
        tin = as_t(surf["tau"][input_idx])
        vin = as_t(surf["iv"][input_idx])[:, None]
        x_in = scale_coords_t(kin, tin)
        y_out = scale_coords_t(as_t(kq), as_t(tq))
        return self.forward_coords(x_in, vin, y_out)

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("DeepONet params:", count_params(DeepSetONet()))
print("GNO params     :", count_params(InterpGNO()))

## v3 — Embedding the SSVI prior in the operator

The audit of v2 produced the central operator result: the graph operator is the accuracy leader
and simultaneously the most arbitrage-dirty model in the study (material butterfly violations on
~24% of the independent audit grid, on 330 of 386 out-of-sample days), while its own penalty-grid
audit reads near zero.

That result is a **diagnosis**. This section adds the **constructive** counterpart, and it closes an
inconsistency in the thesis: the deep smoother of NB03 uses the Ackerer prior–corrector design,

$$w_\theta \;=\; w_{\mathrm{SSVI}} \times \exp(\mathrm{NN}_\theta),$$

and the NB03 ablation shows the prior does the protective work (prior alone 0.15 vp, prior ×
corrector 0.10 vp, corrector without prior 1.06 vp). The operators of v2, by contrast, map the
quote cloud **straight** to the surface with no prior at all, because v2 deliberately replicated
the published Operator Deep Smoothing design in order to audit *that* architecture. Auditing it is
done; the natural next question is whether the same protective mechanism transfers.

**Design.** For every day we fit an SSVI prior from that day's own quotes, calibrated *inside* the
Gatheral–Jacquier region so it carries a printed certificate, and the operator learns only a
bounded multiplicative correction:

$$\widehat\sigma_{\mathrm{IV}}(k,\tau) \;=\; \underbrace{\sigma_{\mathrm{SSVI}}(k,\tau)}_{\text{certified, per day}}
\times \exp\Bigl(\underbrace{s\cdot\tanh\bigl(\mathcal G_\vartheta(a)(k,\tau)\bigr)}_{\text{correction, }|c|\le s}\Bigr).$$

Four properties follow, each of which addresses something v2 exposed.

1. **Training starts exactly at the prior.** The correction head is zero-initialised, so at
   initialisation $c \equiv 0$ and the operator *is* the certified SSVI. Verified numerically below
   to $10^{-8}$.
2. **Positivity is structural, and the dying-softplus failure disappears.** v2 needed a softplus
   output head, which is exactly what collapsed on real data (loss pinned at 1.0000, $\min g$ =
   0.9995, a $k$-flat surface). Here positivity comes from prior $\times \exp(\cdot)$, so the head
   is a plain linear layer and there is no saturating activation to die.
3. **The correction is bounded by construction**, $|c|\le s$. This is a curvature budget in the
   sense of the node-gap bound: it does not *prove* $g\ge0$, but it bounds how far the operator can
   travel from a surface that does.
4. **Inheritance.** Wherever the correction is flat, all derivatives, hence $g$ and
   $\partial_\tau w$, are the prior's, and the prior carries a theorem.

**What this is not.** Prior × correction is a strong inductive bias, *not* a guarantee: a
correction with enough curvature can still break $g \ge 0$. This is reported as a measured
reduction against the v2 baseline under the identical audit, never as a certificate.

**Honesty note on pretraining.** The synthetic bank is generated *from* SSVI, so a fitted prior
recovers it almost exactly and the residual is zero: a prior-embedded model pretrained on it would
have nothing to learn. The prior families therefore pretrain on a **perturbed** synthetic bank
(SSVI times a smooth random bump), so there is a genuine residual to learn. The headline
comparison remains **R1 real-only against R1 real-only**, which is like for like.


In [ ]:
from scipy.optimize import minimize

RUN_PRIOR_FAMILIES = os.environ.get("NB04_RUN_PRIOR", "1") == "1"
CORR_SCALE = float(os.environ.get("NB04_CORR_SCALE", "0.35"))   # |log correction| bound
PRIOR_CACHE = RUN_DIR / "nb04_ssvi_priors.parquet"


def gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi):
    """Endpoint certification (thesis Lemma 'endpoint certification'): on a compact working
    theta-range the two GJ conditions need only be checked at the two endpoints."""
    th_max = alpha * t_hi ** beta
    th_min = alpha * max(t_lo, 1e-6) ** beta
    c1 = eta * th_max ** (1 - gamma) * (1 + abs(rho))
    c2 = eta ** 2 * max(th_max ** (1 - 2 * gamma), th_min ** (1 - 2 * gamma)) * (1 + abs(rho))
    return bool((0 < gamma < 1) and (c1 < 4) and (c2 <= 4)), float(c1), float(c2)


def _clip_ssvi(p):
    rho, eta, gamma, alpha, beta = p
    return (float(np.clip(rho, -0.95, 0.95)), float(max(eta, 1e-4)),
            float(np.clip(gamma, 0.05, 0.95)), float(max(alpha, 1e-6)),
            float(np.clip(beta, 0.30, 1.60)))


def prior_iv_np(k, tau, p):
    """SSVI implied vol at (k, tau) for prior parameters p."""
    tau = np.maximum(np.asarray(tau, float), 1e-12)
    theta = p["alpha"] * tau ** p["beta"]
    w = ssvi_total_variance(np.asarray(k, float), theta, p["rho"], p["eta"], p["gamma"])
    return np.sqrt(np.maximum(w, 1e-12) / tau)

In [ ]:
def fit_ssvi_day(k, tau, iv, maxiter=600):
    """Per-day SSVI prior. IV-target weights (delta method), robust nearest-ATM seed for the
    theta term structure, and a barrier keeping the search inside the GJ region so the fitted
    prior is CERTIFIED rather than luckily arbitrage-free."""
    k = np.asarray(k, float); tau = np.asarray(tau, float); iv = np.asarray(iv, float)
    w = iv ** 2 * tau
    wt = 1.0 / (4.0 * np.maximum(w, 1e-10) * tau)
    wt = wt / wt.mean()
    t_lo, t_hi = float(tau.min()), float(tau.max())
    taus = np.unique(tau)

    tt, tv = [], []
    for t in taus:
        m = tau == t
        kk, ww = k[m], w[m]
        j = int(np.argmin(np.abs(kk)))
        if abs(float(kk[j])) <= 0.20 and float(ww[j]) > 0:
            tt.append(float(t)); tv.append(float(ww[j]))
    if len(tt) >= 2:
        A = np.vstack([np.ones(len(tt)), np.log(tt)]).T
        coef, *_ = np.linalg.lstsq(A, np.log(tv), rcond=None)
        a0, b0 = float(np.exp(coef[0])), float(np.clip(coef[1], 0.4, 1.4))
    else:
        a0, b0 = float(np.median(w / tau ** 0.95)), 0.95

    def sse(p):
        rho, eta, gamma, alpha, beta = _clip_ssvi(p)
        tot = 0.0
        for t in taus:
            m = tau == t
            r_ = ssvi_total_variance(k[m], alpha * t ** beta, rho, eta, gamma) - w[m]
            tot += float(np.sum(wt[m] * r_ * r_))
        ok, _, _ = gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi)
        return tot + (0.0 if ok else 1e3)

    best = None
    for x0 in [(-0.55, 0.9, 0.35, a0, b0), (-0.75, 0.6, 0.45, a0, b0), (-0.30, 1.2, 0.25, a0, b0)]:
        r_ = minimize(sse, x0, method="Nelder-Mead", options={"maxiter": maxiter, "fatol": 1e-14})
        if best is None or r_.fun < best.fun:
            best = r_
    rho, eta, gamma, alpha, beta = _clip_ssvi(best.x)
    p = dict(rho=rho, eta=eta, gamma=gamma, alpha=alpha, beta=beta)
    ok, c1, c2 = gj_ok(rho, eta, gamma, alpha, beta, t_lo, t_hi)
    rmse = float(np.sqrt(np.mean((prior_iv_np(k, tau, p) - iv) ** 2)) * 100)
    return p, dict(gj_ok=ok, gj_c1=c1, gj_c2=c2, prior_rmse_volpts=rmse)

In [ ]:
def attach_priors(bank, cache_path=None, label=""):
    """Fit (or load) one SSVI prior per surface and attach it in place. The per-day prior-only
    RMSE is stored: it is the diagnostic that separates a prior failure from a corrector
    failure, exactly as in the NB03 sparsity study."""
    cache = {}
    if cache_path is not None and Path(cache_path).exists():
        df = pl.read_parquet(cache_path)
        for row in df.iter_rows(named=True):
            cache[row["date"]] = row
    rows, n_fit, t0 = [], 0, time.time()
    for surf in bank:
        key = str(surf["date"])
        if key in cache and key != "synthetic":
            row = cache[key]
            surf["prior"] = {c: float(row[c]) for c in ("rho", "eta", "gamma", "alpha", "beta")}
            surf["prior_diag"] = dict(gj_ok=bool(row["gj_ok"]), gj_c1=float(row["gj_c1"]),
                                      gj_c2=float(row["gj_c2"]),
                                      prior_rmse_volpts=float(row["prior_rmse_volpts"]))
        else:
            p, diag = fit_ssvi_day(surf["k"], surf["tau"], surf["iv"])
            surf["prior"], surf["prior_diag"] = p, diag
            n_fit += 1
        rows.append(dict(date=key, **surf["prior"], **surf["prior_diag"]))
    el = time.time() - t0
    dfr = pl.DataFrame(rows)
    ok_pct = float(dfr["gj_ok"].mean()) * 100
    med = float(dfr["prior_rmse_volpts"].median())
    print(f"[prior:{label}] {len(bank)} surfaces ({n_fit} newly fitted) in {el:.0f}s | "
          f"GJ-certified {ok_pct:.1f}% | median prior-only RMSE {med:.3f} vol pts")
    if cache_path is not None and n_fit:
        dfr.write_parquet(cache_path)
        print("written:", cache_path)
    return dfr

In [ ]:
# ---- perturbed synthetic bank: pure SSVI leaves the corrector nothing to learn ----
def synth_surface_perturbed(r, bump=0.10, **kw):
    """SSVI times a smooth random Gaussian bump in (k, tau). The fitted prior then recovers the
    SSVI part but NOT the bump, so a prior-embedded model has a genuine residual to learn."""
    s = synth_surface(r, **kw)
    k0 = float(r.uniform(K_LO + 0.05, K_HI - 0.05))
    t0 = float(r.uniform(T_LO + 0.1, T_HI - 0.1))
    a = float(r.uniform(-bump, bump))
    wk = float(r.uniform(0.10, 0.30)); wt = float(r.uniform(0.25, 0.80))
    mult = np.exp(a * np.exp(-((s["k"] - k0) / wk) ** 2 - ((s["tau"] - t0) / wt) ** 2))
    s["iv"] = s["iv"] * mult
    s["kind"] = "synthetic_perturbed"
    return s

In [ ]:
if RUN_PRIOR_FAMILIES:
    prior_real = attach_priors(real_bank, cache_path=PRIOR_CACHE, label="real")
    synth_bank_pert = [synth_surface_perturbed(rng) for _ in range(N_SYNTH)]
    _ = attach_priors(synth_bank_pert, label="synth_pert")
    _ = attach_priors(synth_bank, label="synth")

    # prior-only baseline on the test days: the number every prior-embedded model must beat
    _test_prior_rmse = float(np.median([s["prior_diag"]["prior_rmse_volpts"] for s in test_days]))
    print(f"\n[baseline] median prior-only RMSE on the {len(test_days)} test days: "
          f"{_test_prior_rmse:.3f} vol pts  <- the corrector must improve on this")
else:
    synth_bank_pert = synth_bank
    print("[prior] NB04_RUN_PRIOR=0 -- prior-embedded families skipped")

In [ ]:
def prior_iv_t(kq, tq, p):
    return as_t(prior_iv_np(kq, tq, p))


def log_residual_np(k, tau, iv, p):
    """The operator's input feature: how far the quotes sit from the certified prior."""
    return np.log(np.maximum(iv, 1e-8) / np.maximum(prior_iv_np(k, tau, p), 1e-8))

In [ ]:
class PriorDeepONet(DeepSetONet):
    """DeepONet acting on the residual. Differences from the v2 DeepONet:
      - a 4th input channel, the log-residual against the prior;
      - the branch output layer is ZERO-initialised, so the correction is identically 0 at
        initialisation and training starts AT the certified prior;
      - no softplus head: positivity comes from prior * exp(.), which also removes the
        dying-softplus failure mode documented in v2."""
    model_name = "deeponet_prior"

    def __init__(self):
        super().__init__()
        self.encoder = MLP([4, ENC_H, ENC_H, LATENT])
        last = self.psi.net[-1]
        nn.init.zeros_(last.weight)
        nn.init.zeros_(last.bias)

    def encode_surface(self, surf, input_idx):
        p = surf["prior"]
        k = surf["k"][input_idx]; tau = surf["tau"][input_idx]; iv = surf["iv"][input_idx]
        res = log_residual_np(k, tau, iv, p)
        x = torch.cat([scale_coords_t(as_t(k), as_t(tau)),
                       5.0 * as_t(iv)[:, None],
                       as_t(res)[:, None]], dim=-1)
        return self.encoder(x).mean(dim=0)

    def predict_surface(self, surf, kq, tq, input_idx):
        p = surf["prior"]
        z = self.encode_surface(surf, input_idx)
        c = self.psi(z[None, :])[0]
        t = self.trunk(scale_coords_t(as_t(kq), as_t(tq)))
        corr = CORR_SCALE * torch.tanh((t * c).sum(dim=-1))
        return prior_iv_t(kq, tq, p) * torch.exp(corr) + 1e-8

In [ ]:
class PriorGNO(InterpGNO):
    """Graph neural operator acting on the residual. The message passing now carries the
    log-residual instead of the raw IV, and the output head is a zero-initialised linear layer
    (no softplus), so the operator starts at the prior and produces a bounded correction."""
    model_name = "gno_prior"

    def __init__(self, channels=GNO_CHANNELS, J=GNO_LAYERS):
        super().__init__(channels=channels, J=J)
        self.out = nn.Linear(channels, 1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward_coords(self, x_in, v_in, y_out):
        nbrs = build_neighbors(x_in, y_out)
        h_in = self.lift(v_in)
        h = self.layers[0](h_in, x_in, y_out, v_in, nbrs)
        nbrs_out = build_neighbors(y_out, y_out)
        v_zero = torch.zeros(len(y_out), 1, device=y_out.device)
        for layer in self.layers[1:]:
            h = layer(h, y_out, y_out, v_zero, nbrs_out, h_self=h)
        return self.out(h).squeeze(-1)          # unconstrained correction, not an IV

    def predict_surface(self, surf, kq, tq, input_idx):
        p = surf["prior"]
        kin = surf["k"][input_idx]; tin = surf["tau"][input_idx]; ivin = surf["iv"][input_idx]
        res = log_residual_np(kin, tin, ivin, p)
        x_in = scale_coords_t(as_t(kin), as_t(tin))
        y_out = scale_coords_t(as_t(kq), as_t(tq))
        raw = self.forward_coords(x_in, as_t(res)[:, None], y_out)
        corr = CORR_SCALE * torch.tanh(raw)
        return prior_iv_t(kq, tq, p) * torch.exp(corr) + 1e-8

In [ ]:
# ---- assertion: the prior-embedded models start EXACTLY at the certified prior ----
if RUN_PRIOR_FAMILIES:
    _s = test_days[0]
    _idx = np.arange(min(200, len(_s["k"])))
    for _ctor in (PriorDeepONet, PriorGNO):
        _m = _ctor().to(DEVICE).eval()
        with torch.no_grad():
            _out = _m.predict_surface(_s, _s["k"][_idx], _s["tau"][_idx], _idx).cpu().numpy()
        _ref = prior_iv_np(_s["k"][_idx], _s["tau"][_idx], _s["prior"])
        _err = float(np.max(np.abs(_out - _ref)))
        print(f"[init check] {_ctor.model_name:<16} max |pred - prior| = {_err:.2e}  "
              f"params={count_params(_m)}")
        assert _err < 1e-5, "correction is not zero at initialisation: it must start at the prior"
    print("[init check] both prior-embedded operators start at the certified SSVI prior.")

## 4. Training and evaluation

One shared loss (vega-weighted relative fit + butterfly + calendar + curvature regularizers on the penalty grid), one shared training loop, and one evaluator that computes the full audit battery per day.

In [ ]:
def loss_terms(model, surf, r):
    n = len(surf["k"])
    input_idx = sample_idx(n, model.input_points, r)
    fit_idx = sample_idx(n, model.fit_points, r)

    k_fit = surf["k"][fit_idx]
    t_fit = surf["tau"][fit_idx]
    iv_fit_np = surf["iv"][fit_idx]
    iv_fit = as_t(iv_fit_np)
    pred = model.predict_surface(surf, k_fit, t_fit, input_idx)

    vw = vega_weights(as_t(k_fit), as_t(t_fit), iv_fit)
    fit = torch.sqrt(torch.mean(vw * ((pred - iv_fit) / torch.clamp(iv_fit, min=1e-6)) ** 2))

    V = model.predict_surface(surf, KG_FLAT, TG_FLAT, input_idx).reshape(GRID_NT, GRID_NK)
    W = V ** 2 * TTG_T
    Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
    Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
    G = durrleman_g_t(KKG_T[:, 1:-1], W[:, 1:-1], Wk, Wkk)

    but = torch.relu(EPS_BUT - G).mean()
    cal = torch.relu(-(W[1:, :] - W[:-1, :])).mean()
    reg_t = torch.sqrt(torch.mean((V[2:, :] - 2 * V[1:-1, :] + V[:-2, :]) ** 2) + 1e-12)
    reg_k = torch.sqrt(torch.mean((V[:, 2:] - 2 * V[:, 1:-1] + V[:, :-2]) ** 2) + 1e-12)
    total = LAMBDAS["fit"] * fit + LAMBDAS["but"] * but + LAMBDAS["cal"] * cal + LAMBDAS["reg_t"] * reg_t + LAMBDAS["reg_k"] * reg_k
    return total, dict(fit=fit, but=but, cal=cal, reg_t=reg_t, reg_k=reg_k)

In [ ]:
def batches(bank, batch_size, r):
    idx = r.permutation(len(bank))
    for start in range(0, len(idx), batch_size):
        yield [bank[i] for i in idx[start:start + batch_size]]


def train_full_pass(model, bank, epochs, lr, tag, eval_bank=None):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    r = np.random.default_rng(SEED + abs(hash(tag)) % 1_000_000)
    history = []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        ep_losses = []
        ep_terms = []
        for batch in batches(bank, model.batch_size, r):
            opt.zero_grad(set_to_none=True)
            losses, term_rows = [], []
            for surf in batch:
                loss, terms = loss_terms(model, surf, r)
                losses.append(loss)
                term_rows.append({k: float(v.detach().cpu()) for k, v in terms.items()})
            batch_loss = torch.stack(losses).mean()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            ep_losses.append(float(batch_loss.detach().cpu()))
            ep_terms.extend(term_rows)

        row = dict(model=model.model_name, tag=tag, epoch=ep, loss=float(np.mean(ep_losses)), seconds=time.time() - t0)
        for key in ["fit", "but", "cal", "reg_t", "reg_k"]:
            row[key] = float(np.mean([x[key] for x in ep_terms]))
        if eval_bank is not None and (ep == 1 or ep == epochs or ep % max(1, epochs // 5) == 0):
            ev = evaluate_model(model, eval_bank, max_days=min(20, len(eval_bank)), tag=f"{tag}_val")
            row.update({f"val_{k}": v for k, v in ev.items() if isinstance(v, (int, float))})
        history.append(row)

        if ep == 1 or ep == epochs or ep % max(1, epochs // 5) == 0:
            msg = f"[{tag}] epoch {ep:>3}/{epochs} full-pass loss={row['loss']:.4f}"
            if "val_rmse_volpts" in row:
                msg += f" | val rmse={row['val_rmse_volpts']:.3f} vol pts"
            msg += f" | elapsed={row['seconds']:.0f}s"
            print(msg)
    return model, history

In [ ]:
@torch.no_grad()
def predict_np(model, surf, kq, tq, input_points=None):
    model.eval()
    n = len(surf["k"])
    m = input_points or model.input_points
    if n > m:
        idx = np.linspace(0, n - 1, m).round().astype(int)
    else:
        idx = np.arange(n)
    out = model.predict_surface(surf, np.asarray(kq), np.asarray(tq), idx)
    return out.detach().cpu().numpy()


def holdout_split(surf):
    """NB03 protocol: 20% per-day holdout with a crc32(date)-seeded rng, so NB03's and NB04's
    holdout numbers are computed on the SAME held-out quotes."""
    r = np.random.default_rng(zlib.crc32(str(surf["date"]).encode()))
    hold = r.random(len(surf["k"])) < HOLDOUT_FRAC
    if hold.all() or (~hold).sum() < 20:
        hold[:] = False
    return hold


def _nanmean(xs):
    xs = [x for x in xs if x == x]           # drop NaN
    return float(np.mean(xs)) if xs else float("nan")


def _nanmin(xs):
    xs = [x for x in xs if x == x]
    return float(np.min(xs)) if xs else float("nan")

In [ ]:
@torch.no_grad()
def evaluate_model(model, days, max_days=None, tag="eval"):
    """v3 [N4-1, N4-3]: per-day hull audit alongside the box audit, and fit-gated arbitrage
    aggregates (days with quote RMSE <= FIT_GATE_VP). 'Never report a metric without a
    validity gate behind it': the ungated columns keep continuity with v2; the gated columns
    are the ones that survive the retort 'the violating days are just bad fits'."""
    use_days = days if max_days is None else days[:max_days]
    day_rows = []
    for surf in use_days:
        pred = predict_np(model, surf, surf["k"], surf["tau"])
        err = pred - surf["iv"]
        rmse = float(np.sqrt(np.mean(err ** 2))) * 100
        hold = holdout_split(surf)
        hold_rmse = float("nan")
        if hold.any():
            kept = dict(date=surf["date"], k=surf["k"][~hold], tau=surf["tau"][~hold],
                        iv=surf["iv"][~hold], kind=surf.get("kind", "real"),
                        prior=surf.get("prior"))
            ph = predict_np(model, kept, surf["k"][hold], surf["tau"][hold])
            hold_rmse = float(np.sqrt(np.mean((ph - surf["iv"][hold]) ** 2))) * 100
        Vp = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        pg = fd_audit_from_grid(Vp)
        Va = predict_np(model, surf, KA_FLAT, TA_FLAT).reshape(AUDIT_NT, AUDIT_NK)
        ia = independent_audit(Va, hull=hull_mask_for(surf))
        bf = blind_flags(pg, ia)
        day_rows.append(dict(
            rmse=rmse, hold_rmse=hold_rmse,
            mae=float(np.mean(np.abs(err))) * 100,
            rel=float(np.sqrt(np.mean((err / surf["iv"]) ** 2))),
            fit_ok=bool(rmse <= FIT_GATE_VP),
            pg=pg, ia=ia, bf=bf,
        ))

    ok = [d for d in day_rows if d["fit_ok"]]
    holds = [d["hold_rmse"] for d in day_rows if d["hold_rmse"] == d["hold_rmse"]]
    out = dict(
        tag=tag, n_days=len(use_days),
        rmse_volpts=float(np.mean([d["rmse"] for d in day_rows])),
        rmse_sd_volpts=float(np.std([d["rmse"] for d in day_rows])),
        rmse_holdout_volpts=float(np.mean(holds)) if holds else None,
        mae_volpts=float(np.mean([d["mae"] for d in day_rows])),
        rel_rmse=float(np.mean([d["rel"] for d in day_rows])),
        # penalty-grid view (what the optimizer saw)
        min_g=_nanmin([d["pg"]["min_g"] for d in day_rows]),
        butterfly_viol_pct=_nanmean([d["pg"]["butterfly_viol_pct"] for d in day_rows]),
        calendar_viol_pct=_nanmean([d["pg"]["calendar_viol_pct"] for d in day_rows]),
        pen_bfly_viol_mat_pct=_nanmean([d["pg"]["bfly_viol_mat_pct"] for d in day_rows]),
        pen_cal_viol_mat_pct=_nanmean([d["pg"]["cal_viol_mat_pct"] for d in day_rows]),
        # independent audit, full box
        audit_min_g=_nanmin([d["ia"]["audit_min_g"] for d in day_rows]),
        audit_min_dtw=_nanmin([d["ia"]["audit_min_dtw"] for d in day_rows]),
        audit_bfly_viol_pct=_nanmean([d["ia"]["audit_bfly_viol_pct"] for d in day_rows]),
        audit_cal_viol_pct=_nanmean([d["ia"]["audit_cal_viol_pct"] for d in day_rows]),
        audit_bfly_viol_mat_pct=_nanmean([d["ia"]["audit_bfly_viol_mat_pct"] for d in day_rows]),
        audit_cal_viol_mat_pct=_nanmean([d["ia"]["audit_cal_viol_mat_pct"] for d in day_rows]),
        # independent audit, quoted domain (hull)
        hull_frac=_nanmean([d["ia"]["hull_frac"] for d in day_rows]),
        hull_min_g=_nanmin([d["ia"]["hull_min_g"] for d in day_rows]),
        hull_min_dtw=_nanmin([d["ia"]["hull_min_dtw"] for d in day_rows]),
        hull_bfly_viol_mat_pct=_nanmean([d["ia"]["hull_bfly_viol_mat_pct"] for d in day_rows]),
        hull_cal_viol_mat_pct=_nanmean([d["ia"]["hull_cal_viol_mat_pct"] for d in day_rows]),
        # blind-spot day counts (box, material-vs-material)
        blind_bfly_days=int(sum(d["bf"]["blind_bfly"] for d in day_rows)),
        blind_cal_days=int(sum(d["bf"]["blind_cal"] for d in day_rows)),
        # fit-gated aggregates
        n_fit_ok=len(ok),
        gated_audit_bfly_viol_mat_pct=_nanmean([d["ia"]["audit_bfly_viol_mat_pct"] for d in ok]),
        gated_audit_cal_viol_mat_pct=_nanmean([d["ia"]["audit_cal_viol_mat_pct"] for d in ok]),
        gated_hull_bfly_viol_mat_pct=_nanmean([d["ia"]["hull_bfly_viol_mat_pct"] for d in ok]),
        gated_hull_cal_viol_mat_pct=_nanmean([d["ia"]["hull_cal_viol_mat_pct"] for d in ok]),
        gated_blind_bfly_days=int(sum(d["bf"]["blind_bfly"] for d in ok)),
        gated_blind_cal_days=int(sum(d["bf"]["blind_cal"] for d in ok)),
    )
    n_blind_b, n_blind_c = out["blind_bfly_days"], out["blind_cal_days"]
    if n_blind_b or n_blind_c:
        print(f"  !! BLIND SPOT [{tag}]: {n_blind_b} day(s) butterfly / {n_blind_c} day(s) "
              f"calendar -- materially clean on the PENALTY grid, materially violating on the "
              f"independent audit grid. The operator inherits the collocation blind spot.")
    if max_days is None:
        print(f"  >> domain split [{tag}]: material bfly {out['audit_bfly_viol_mat_pct']:.2f}% (box) "
              f"vs {out['hull_bfly_viol_mat_pct']:.2f}% (quoted domain) | material cal "
              f"{out['audit_cal_viol_mat_pct']:.2f}% (box) vs {out['hull_cal_viol_mat_pct']:.2f}% "
              f"(quoted domain) | hull covers {out['hull_frac']*100:.0f}% of the box | "
              f"fit gate passed on {out['n_fit_ok']}/{out['n_days']} days.")
    return out

In [ ]:
def save_history(rows, name):
    path = RUN_DIR / name
    pl.DataFrame(rows).write_parquet(path)
    print("written:", path)


def run_family(model_ctor, family_name, lr_pre, lr_real, lr_ft, pretrain_bank=None):
    # v3: the pretraining bank is now an argument. The prior-embedded families pretrain on
    # PERTURBED synthetics: on pure SSVI synthetics the fitted prior IS the ground truth,
    # the residual is zero, and the corrector would have nothing to learn.
    pretrain_bank = synth_bank if pretrain_bank is None else pretrain_bank
    histories = []
    results = []
    models = {}
    print("\n" + "=" * 80)
    print("MODEL FAMILY:", family_name)
    print("=" * 80)

    # v2: RESUME mode -- reload the saved weights and re-run only evaluation/audit/export.
    # Lets the (new) independent audit and exports run on the laptop without the 6h retrain.
    bundle_path = RUN_DIR / f"nb04_{family_name}_state_dicts.pt"
    if RESUME and bundle_path.exists():
        print(f"[resume] loading {bundle_path} -- skipping training")
        bundle = torch.load(bundle_path, map_location=DEVICE)
        for regime, sd in bundle.items():
            m = model_ctor().to(DEVICE)
            m.load_state_dict(sd)
            models[regime] = copy.deepcopy(m).cpu()
            tag = f"{family_name}_{regime.replace(' ', '_').replace('+', '_')}"
            res = evaluate_model(m, test_days, tag=tag)
            results.append(dict(model=family_name, regime=regime, **res))
        return histories, results, models

    print("\nR2: synthetic-only")
    m_pre = model_ctor().to(DEVICE)
    print("params:", count_params(m_pre))
    m_pre, h = train_full_pass(m_pre, pretrain_bank, EPOCHS_PRE, lr_pre, f"{family_name}_R2_pretrain", eval_bank=val_days)
    histories += h
    models["R2 synth-only"] = copy.deepcopy(m_pre).cpu()
    res = evaluate_model(m_pre, test_days, tag=f"{family_name}_R2_synth_only")
    results.append(dict(model=family_name, regime="R2 synth-only", **res))

    print("\nR1: real-only")
    m_r1 = model_ctor().to(DEVICE)
    print("params:", count_params(m_r1))
    m_r1, h = train_full_pass(m_r1, train_days, EPOCHS_R1, lr_real, f"{family_name}_R1_real", eval_bank=val_days)
    histories += h
    models["R1 real-only"] = copy.deepcopy(m_r1).cpu()
    res = evaluate_model(m_r1, test_days, tag=f"{family_name}_R1_real_only")
    if res["rel_rmse"] > 0.95:
        print(f"  !! COLLAPSE GUARD [{family_name} R1]: rel_rmse = {res['rel_rmse']:.3f} ~ 1.0 "
              f"means the model predicts ~0 everywhere. ALL downstream metrics for this cell "
              f"(including its trivially-perfect discretization invariance) are VACUOUS.")
    results.append(dict(model=family_name, regime="R1 real-only", **res))

    print("\nR3: synthetic pretrain + real finetune")
    m_r3 = copy.deepcopy(m_pre).to(DEVICE)
    m_r3, h = train_full_pass(m_r3, train_days, EPOCHS_FT, lr_ft, f"{family_name}_R3_finetune", eval_bank=val_days)
    histories += h
    models["R3 pretrain+finetune"] = copy.deepcopy(m_r3).cpu()
    res = evaluate_model(m_r3, test_days, tag=f"{family_name}_R3_pretrain_finetune")
    results.append(dict(model=family_name, regime="R3 pretrain+finetune", **res))

    save_history(histories, f"nb04_{family_name}_training_curves.parquet")
    state_bundle = {
        regime: model.state_dict()
        for regime, model in models.items()
    }
    torch.save(state_bundle, RUN_DIR / f"nb04_{family_name}_state_dicts.pt")
    print("written:", RUN_DIR / f"nb04_{family_name}_state_dicts.pt")
    return histories, results, models

In [ ]:
all_histories, all_results, trained_models = [], [], {}

deep_hist, deep_res, deep_models = run_family(DeepSetONet, "deeponet", lr_pre=3e-3, lr_real=5e-4, lr_ft=1e-3)
all_histories += deep_hist
all_results += deep_res
trained_models["deeponet"] = deep_models

gno_hist, gno_res, gno_models = run_family(InterpGNO, "gno", lr_pre=1e-3, lr_real=1e-3, lr_ft=5e-4)
all_histories += gno_hist
all_results += gno_res
trained_models["gno"] = gno_models

# --- v3: prior-embedded variants (the constructive answer to the audit result) ---
if RUN_PRIOR_FAMILIES:
    pdeep_hist, pdeep_res, pdeep_models = run_family(
        PriorDeepONet, "deeponet_prior", lr_pre=3e-3, lr_real=5e-4, lr_ft=1e-3,
        pretrain_bank=synth_bank_pert)
    all_histories += pdeep_hist; all_results += pdeep_res
    trained_models["deeponet_prior"] = pdeep_models

    pgno_hist, pgno_res, pgno_models = run_family(
        PriorGNO, "gno_prior", lr_pre=1e-3, lr_real=1e-3, lr_ft=5e-4,
        pretrain_bank=synth_bank_pert)
    all_histories += pgno_hist; all_results += pgno_res
    trained_models["gno_prior"] = pgno_models

In [ ]:
summary_path = RUN_DIR / "nb04_operator_summary.parquet"
hist_path = RUN_DIR / "nb04_all_training_curves.parquet"
pl.DataFrame(all_results).write_parquet(summary_path)
pl.DataFrame(all_histories).write_parquet(hist_path)
print("written:", summary_path)
print("written:", hist_path)
pl.DataFrame(all_results)

## 5. Results

Per-day metrics, regime table, invariance study, then the standard figures.

In [ ]:
@torch.no_grad()
def per_day_metrics(model, model_name, regime, days):
    rows = []
    model = model.to(DEVICE)
    for surf in days:
        pred = predict_np(model, surf, surf["k"], surf["tau"])
        err = pred - surf["iv"]
        rmse = float(np.sqrt(np.mean(err ** 2)) * 100)
        hold = holdout_split(surf)
        hold_rmse = None
        if hold.any():
            kept = dict(date=surf["date"], k=surf["k"][~hold], tau=surf["tau"][~hold],
                        iv=surf["iv"][~hold], kind=surf.get("kind", "real"),
                        prior=surf.get("prior"))
            ph = predict_np(model, kept, surf["k"][hold], surf["tau"][hold])
            hold_rmse = float(np.sqrt(np.mean((ph - surf["iv"][hold]) ** 2)) * 100)
        Vp = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        pg = fd_audit_from_grid(Vp)
        Va = predict_np(model, surf, KA_FLAT, TA_FLAT).reshape(AUDIT_NT, AUDIT_NK)
        ia = independent_audit(Va, hull=hull_mask_for(surf))
        bf = blind_flags(pg, ia)
        rows.append(dict(
            model=model_name,
            regime=regime,
            date=str(surf["date"]),
            n_quotes=int(len(surf["k"])),
            iv_level=float(np.median(surf["iv"])),
            rmse_volpts=rmse,
            rmse_holdout_volpts=hold_rmse,
            mae_volpts=float(np.mean(np.abs(err)) * 100),
            rel_rmse=float(np.sqrt(np.mean((err / surf["iv"]) ** 2))),
            fit_ok=bool(rmse <= FIT_GATE_VP),
            **pg, **ia, **bf,
        ))
    return rows

In [ ]:
per_day_rows = []
for model_name, regimes in trained_models.items():
    for regime, model in regimes.items():
        per_day_rows.extend(per_day_metrics(model, model_name, regime, test_days))

per_day = pl.DataFrame(per_day_rows)
q1, q2 = per_day.select(pl.col("iv_level").quantile(0.33)).item(), per_day.select(pl.col("iv_level").quantile(0.66)).item()
per_day = per_day.with_columns(
    pl.when(pl.col("iv_level") <= q1).then(pl.lit("low_iv"))
    .when(pl.col("iv_level") <= q2).then(pl.lit("mid_iv"))
    .otherwise(pl.lit("high_iv"))
    .alias("regime_bucket")
)
per_day_path = RUN_DIR / "nb04_operator_per_day.parquet"
per_day.write_parquet(per_day_path)
print("written:", per_day_path)
per_day.head()

In [ ]:
def subsurf(surf, frac, r):
    n = len(surf["k"])
    m = max(8, int(frac * n))
    idx = np.sort(r.choice(n, size=min(m, n), replace=False))
    return dict(date=surf["date"], k=surf["k"][idx], tau=surf["tau"][idx],
                iv=surf["iv"][idx], kind=surf.get("kind", "real"), prior=surf.get("prior"))


@torch.no_grad()
def invariance_rows(model, model_name, regime, days, fracs=(1.0, 0.75, 0.50, 0.25), n_days=10):
    """v2: invariance is only reported for days the model actually FITS (rmse < FIT_GATE_VP).
    A collapsed/constant model is trivially discretization-invariant -- the v1 plot showed
    deeponet R1 at a perfect 0.0 shift for exactly that vacuous reason. Same lesson as the
    NB03 stress test: gate the metric on fit validity before reading it."""
    r = np.random.default_rng(SEED + 99)
    rows = []
    model = model.to(DEVICE)
    for surf in days[:min(n_days, len(days))]:
        pred = predict_np(model, surf, surf["k"], surf["tau"])
        day_rmse = float(np.sqrt(np.mean((pred - surf["iv"]) ** 2)) * 100)
        if day_rmse > FIT_GATE_VP:
            rows.append(dict(model=model_name, regime=regime, date=str(surf["date"]),
                             input_frac=None, n_input=len(surf["k"]),
                             mean_abs_surface_shift_volpts=None,
                             max_abs_surface_shift_volpts=None,
                             gated_out=True, day_rmse_volpts=day_rmse))
            continue
        base = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        for frac in fracs:
            s2 = subsurf(surf, frac, r)
            V = predict_np(model, s2, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
            rows.append(dict(
                model=model_name, regime=regime, date=str(surf["date"]),
                input_frac=float(frac), n_input=int(len(s2["k"])),
                mean_abs_surface_shift_volpts=float(np.mean(np.abs(V - base)) * 100),
                max_abs_surface_shift_volpts=float(np.max(np.abs(V - base)) * 100),
                gated_out=False, day_rmse_volpts=day_rmse,
            ))
    return rows

In [ ]:
inv = []
for model_name, regimes in trained_models.items():
    for regime, model in regimes.items():
        inv.extend(invariance_rows(model, model_name, regime, test_days))

inv_df = pl.DataFrame(inv)
n_gated = inv_df.filter(pl.col("gated_out")).height if "gated_out" in inv_df.columns else 0
if n_gated:
    print(f"[gate] {n_gated} day/model cells excluded from the invariance study "
          f"(day RMSE > {FIT_GATE_VP} vp): invariance of a model that cannot fit is vacuous.")
inv_df = inv_df.filter(~pl.col("gated_out")) if "gated_out" in inv_df.columns else inv_df
inv_path = RUN_DIR / "nb04_operator_invariance.parquet"
inv_df.write_parquet(inv_path)
print("written:", inv_path)
inv_df.head()

In [ ]:
summary = pl.read_parquet(RUN_DIR / "nb04_operator_summary.parquet")
print(summary.sort(["model", "rmse_volpts"]))

fig = go.Figure()
for model_name in summary["model"].unique().to_list():
    sub = summary.filter(pl.col("model") == model_name).sort("regime")
    fig.add_trace(go.Bar(
        name=model_name,
        x=sub["regime"].to_list(),
        y=sub["rmse_volpts"].to_list(),
        error_y=dict(array=sub["rmse_sd_volpts"].to_list()),
    ))
fig.update_layout(
    width=950, height=460, barmode="group",
    title="NB04 test performance: operator families and transfer regimes",
    yaxis_title="Test IV RMSE (vol points)",
    xaxis_title="training regime",
)
fig.show()

In [ ]:
per_day = pl.read_parquet(RUN_DIR / "nb04_operator_per_day.parquet")
regime_table = (
    per_day.group_by(["model", "regime", "regime_bucket"])
    .agg(
        pl.col("rmse_volpts").mean().alias("mean_rmse_volpts"),
        # v3: the table the thesis quotes uses the INDEPENDENT audit (box + quoted domain),
        # not the penalty-grid view the optimizer saw.
        pl.col("audit_bfly_viol_mat_pct").mean().alias("mean_audit_bfly_mat_pct"),
        pl.col("audit_cal_viol_mat_pct").mean().alias("mean_audit_cal_mat_pct"),
        pl.col("hull_bfly_viol_mat_pct").mean().alias("mean_hull_bfly_mat_pct"),
        pl.col("hull_cal_viol_mat_pct").mean().alias("mean_hull_cal_mat_pct"),
        pl.col("blind_bfly").sum().alias("blind_bfly_days"),
        pl.col("blind_cal").sum().alias("blind_cal_days"),
        pl.col("fit_ok").sum().alias("n_fit_ok"),
        pl.len().alias("n_days"),
    )
    .sort(["model", "regime", "regime_bucket"])
)
regime_table_path = RUN_DIR / "nb04_operator_regime_table.parquet"
regime_table.write_parquet(regime_table_path)
print("written:", regime_table_path)
print(regime_table)

In [ ]:
inv_df = pl.read_parquet(RUN_DIR / "nb04_operator_invariance.parquet")
inv_mean = (
    inv_df.group_by(["model", "regime", "input_frac"])
    .agg(pl.col("mean_abs_surface_shift_volpts").mean().alias("shift"))
    .sort(["model", "regime", "input_frac"])
)

fig = go.Figure()
for model_name in inv_mean["model"].unique().to_list():
    for regime in inv_mean.filter(pl.col("model") == model_name)["regime"].unique().to_list():
        sub = inv_mean.filter((pl.col("model") == model_name) & (pl.col("regime") == regime)).sort("input_frac")
        fig.add_trace(go.Scatter(
            x=(sub["input_frac"] * 100).to_list(),
            y=sub["shift"].to_list(),
            mode="lines+markers",
            name=f"{model_name} | {regime}",
        ))
fig.update_layout(
    width=950, height=470,
    title="Discretization invariance: dense surface shift under input subsampling",
    xaxis_title="% of input quotes retained",
    yaxis_title="mean absolute surface shift (vol points)",
)
fig.update_xaxes(autorange="reversed")
fig.show()

In [ ]:
best_row = summary.sort("rmse_volpts").row(0, named=True)
best_model_name, best_regime = best_row["model"], best_row["regime"]
model = trained_models[best_model_name][best_regime].to(DEVICE)
surf = test_days[-1]

fig = go.Figure()
taus = np.unique(np.round(surf["tau"], 3))
sel = taus[np.linspace(0, len(taus) - 1, min(5, len(taus))).round().astype(int)]
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
for tau, color in zip(sel, colors):
    m = np.isclose(np.round(surf["tau"], 3), tau)
    if m.sum() < 4:
        continue
    kk = np.linspace(surf["k"][m].min(), surf["k"][m].max(), 150)
    vv = predict_np(model, surf, kk, np.full_like(kk, tau))
    fig.add_trace(go.Scatter(x=surf["k"][m], y=surf["iv"][m], mode="markers", marker=dict(color=color, size=6), name=f"{tau*365:.0f}d quotes"))
    fig.add_trace(go.Scatter(x=kk, y=vv, line=dict(color=color), showlegend=False))
fig.update_layout(
    width=920, height=450,
    title=f"Best NB04 operator on unseen day {surf['date']}: {best_model_name}, {best_regime}",
    xaxis_title="log-moneyness k", yaxis_title="IV",
)
fig.show()

V = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
fig = go.Figure(go.Surface(x=KG, y=TG, z=V, colorscale="Viridis", colorbar=dict(title="IV")))
fig.update_layout(width=780, height=520, title="Operator-smoothed IV surface", scene=dict(xaxis_title="k", yaxis_title="tau", zaxis_title="IV"))
fig.show()

W = V ** 2 * TTG
Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
G = durrleman_g_np(KKG[:, 1:-1], W[:, 1:-1], Wk, Wkk)
fig = go.Figure(go.Heatmap(z=G, x=KG[1:-1], y=TG, zmid=0, colorscale="RdBu", colorbar=dict(title="g")))
fig.update_layout(width=760, height=420, title="Butterfly audit: Durrleman g(k,tau)", xaxis_title="k", yaxis_title="tau")
fig.show()

## 6. Latency and exports for NB05

In [ ]:
@torch.no_grad()
def latency_table(model, surf, sizes=(200, 600, 1525)):
    model = model.to(DEVICE).eval()
    rows = []
    for m in sizes:
        kq = np.linspace(K_LO + 0.02, K_HI - 0.02, m)
        tq = np.full(m, 0.5)
        # warmup + timed
        predict_np(model, surf, kq, tq)
        t0 = time.time()
        for _ in range(5):
            predict_np(model, surf, kq, tq)
        rows.append(dict(model=model.model_name, n_eval_points=m,
                         ms_per_surface=(time.time() - t0) / 5 * 1000, device=str(DEVICE)))
    return rows


lat_rows = []
for family, regimes in trained_models.items():
    best = regimes.get("R3 pretrain+finetune") or list(regimes.values())[-1]
    lat_rows += latency_table(best, test_days[-1])
lat_df = pl.DataFrame(lat_rows)
lat_df.write_parquet(RUN_DIR / "nb04_latency.parquet")
print(lat_df)
print("Context: the per-day deep smoother (NB03) costs ~80-100 s/day; SVI ~1 s/day. The operator "
      "amortizes training across ALL days -- this table is its selling point, quantified.")

In [ ]:
# Dense w on a fine grid per test day for the best model of each family. NB05 differentiates by
# validated finite differences on this fine grid (NB03 exports exact-autodiff fields; the FD-vs-
# exact asymmetry is documented there and bounded by the FD validation above).
SURF_DIR = OUT_DIR / "nb04_surfaces"
SURF_DIR.mkdir(parents=True, exist_ok=True)
# 161x61: sized so that finite-difference g and d_tau w errors on the export grid are both
# <= VIOL_TOL_MAT/3 (measured against analytic SSVI: g ~ 2.9e-4 with dk^2 scaling, wt ~ 2.9e-4
# with dt^2 scaling). Coarser grids let differentiation noise masquerade as violations.
EXPORT_NK, EXPORT_NT = 321, 61   # k uniform x tau HYBRID (exp U uniform, ~119 nodes):
                                 # NB05 reconstructs (wt, g) from w by finite differences on this
                                 # grid; the hybrid short-end density keeps |wt err| ~ 3e-5 and
                                 # |g err| ~ 1e-4, both << VIOL_TOL_MAT (validated in NB05).
KE = np.linspace(K_LO + 0.01, K_HI - 0.01, EXPORT_NK)
_te = np.exp(np.linspace(np.log(T_LO + 0.01), np.log(T_HI - 0.01), EXPORT_NT))
_tu = np.linspace(T_LO + 0.01, T_HI - 0.01, EXPORT_NT)
TE = np.unique(np.round(np.concatenate([_te, _tu]), 10))
KKE, TTE = np.meshgrid(KE, TE)
EXPORT_DAYS = int(os.environ.get("NB04_EXPORT_DAYS", "386"))   # last N test days; raise for prod

export_index = []
for family, regimes in trained_models.items():
    # export the best-holdout regime of each family
    fam_rows = [r for r in all_results if r["model"] == family and r.get("rmse_holdout_volpts")]
    if not fam_rows:
        continue
    best_regime = min(fam_rows, key=lambda r: r["rmse_holdout_volpts"])["regime"]
    model = regimes[best_regime].to(DEVICE)
    for surf in test_days[-EXPORT_DAYS:]:
        V = predict_np(model, surf, KKE.ravel(), TTE.ravel()).reshape(len(TE), EXPORT_NK)
        W = V ** 2 * TTE
        fn = SURF_DIR / f"{family}_{best_regime.split()[0]}_{surf['date']}.npz"
        np.savez_compressed(fn, k=KE, tau=TE, w=W, iv=V,
                            model=family, tag=best_regime, date=str(surf["date"]))
        export_index.append(dict(model=family, regime=best_regime,
                                 date=str(surf["date"]), path=str(fn)))
if export_index:
    pl.DataFrame(export_index).write_parquet(SURF_DIR / "index_operator.parquet")
    print(f"exported {len(export_index)} operator surface pack(s) -> {SURF_DIR}")

In [ ]:
run_config = dict(
    seed=SEED,
    device=str(DEVICE),
    real_parquet=str(REAL_PARQUET),
    n_real_days=len(real_bank),
    n_synth=N_SYNTH,
    train_days=len(train_days),
    val_days=len(val_days),
    test_days=len(test_days),
    epochs_pre=EPOCHS_PRE,
    epochs_r1=EPOCHS_R1,
    epochs_ft=EPOCHS_FT,
    max_quotes_load=MAX_QUOTES_LOAD,
    fit_points_deep=FIT_POINTS_DEEP,
    input_points_deep=INPUT_POINTS_DEEP,
    fit_points_gno=FIT_POINTS_GNO,
    input_points_gno=INPUT_POINTS_GNO,
    batch_deep=BATCH_DEEP,
    batch_gno=BATCH_GNO,
    latent=LATENT,
    enc_h=ENC_H,
    trunk_h=TRUNK_H,
    gno_channels=GNO_CHANNELS,
    gno_layers=GNO_LAYERS,
    gno_k=GNO_K,
    grid_nk=GRID_NK,
    grid_nt=GRID_NT,
    audit_nk=AUDIT_NK,
    audit_nt=AUDIT_NT,
    viol_tol_mat=VIOL_TOL_MAT,
    holdout_frac=HOLDOUT_FRAC,
    fit_gate_vp=FIT_GATE_VP,
    lambdas=LAMBDAS,
)
config_path = RUN_DIR / "nb04_run_config.json"
config_path.write_text(json.dumps(run_config, indent=2))
print("written:", config_path)
print(json.dumps(run_config, indent=2))